# PyTorch 训练模板（Dataset → DataLoader → 模型 → 损失/优化器 → 训练 → 可视化）

本 notebook 与 `Dataset_and_DataLoader.ipynb` **同一套骨架**，方便你换数据、换网络时只改局部。

## 五段式工作流

1. **数据**：`Dataset` 描述「第 i 条样本是什么」；`DataLoader` 负责 batch、shuffle、多进程加载。
2. **模型**：继承 `nn.Module`，在 `forward` 里写从 `x` 到输出的计算图。
3. **损失与优化器**：`criterion` 比较预测与标签；`optimizer` 根据 `loss.backward()` 更新参数。
4. **训练**：双层循环 —— `epoch` × `batch`；每步：`zero_grad` → `forward` → `loss` → `backward` → `step`。
5. **可视化**：把每个 epoch 的标量存进 list，用 `matplotlib` 画曲线。

## 建议你动手改的地方（学习顺序）

- 先改 **`batch_size` / `lr` / `num_epochs`**，观察 loss 曲线变化。
- 再改 **`Model` 的层数或隐藏维度**，看是否过拟合/欠拟合。
- 换任务时：重写 **`Dataset.__init__` / `__getitem__`**，并相应改 `forward` 的输入输出维度与 **损失函数**。

## 0. 依赖与全局配置

In [ ]:
import os
import urllib.request
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# ---------- 可改：随机种子，便于复现 ----------
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# ---------- 可改：训练超参数 ----------
batch_size = 64
num_epochs = 2000
learning_rate = 0.001

# device：有 GPU 用 cuda，否则 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. Dataset + DataLoader（数据准备）

- `__len__`：样本总数。
- `__getitem__(index)`：返回一对 `(特征张量, 标签张量)`，形状需与后面 `model` / `criterion` 一致。
- 下面示例使用 Pima 糖尿病二分类数据（与参考 notebook 一致）。

In [ ]:
class DiabetesDataset(Dataset):
    """从 csv 读入；特征标准化；转为 tensor 供 DataLoader 按索引取样。"""

    def __init__(self, filepath: str):
        xy = np.loadtxt(filepath, delimiter=",", dtype=np.float32)
        self.len = xy.shape[0]
        x_np = xy[:, :-1]
        y_np = xy[:, [-1]]
        mean = x_np.mean(axis=0)
        std = x_np.std(axis=0)
        x_np = (x_np - mean) / (std + 1e-7)
        self.x_data = torch.from_numpy(x_np)
        self.y_data = torch.from_numpy(y_np)

    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]

    def __len__(self):
        return self.len


data_path = "diabetes.csv"
if not os.path.exists(data_path):
    url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
    urllib.request.urlretrieve(url, data_path)
    print("Downloaded:", data_path)

dataset = DiabetesDataset(data_path)
print("X:", dataset.x_data.shape, "y:", dataset.y_data.shape)

train_loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=(device.type == "cuda"),
    num_workers=0,
)

## 2. 模型（`nn.Module`）

- 第一层 `Linear` 的 **in_features** 必须等于每个样本的特征维（此处为 8）。
- 二分类若最后一层 **不加 sigmoid**，应配合 `BCEWithLogitsLoss`。

In [ ]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(8, 6)
        self.linear2 = nn.Linear(6, 4)
        self.linear3 = nn.Linear(4, 2)
        self.linear4 = nn.Linear(2, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.linear1(x))
        x = self.relu(self.linear2(x))
        x = self.relu(self.linear3(x))
        x = self.linear4(x)
        return x


model = Model().to(device)

## 3. 损失函数与优化器

- `BCEWithLogitsLoss`：内部对 logits 做 sigmoid，数值比「sigmoid + BCELoss」更稳。
- `Adam`：常用默认选择；也可试 `SGD(momentum=0.9)`。

In [ ]:
criterion = nn.BCEWithLogitsLoss(reduction="mean")
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

## 4. 训练循环

- 外层：`for epoch in range(num_epochs)`。
- 内层：`for batch_x, batch_y in train_loader`，每个 batch 搬到 `device`。
- 记录整轮平均 loss / acc，便于画图。

In [ ]:
loss_history = []
acc_history = []

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    epoch_correct = 0
    n_samples = 0

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device, non_blocking=True)
        batch_y = batch_y.to(device, non_blocking=True)

        logits = model(batch_x)
        loss = criterion(logits, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            pred = (logits >= 0).float()
            epoch_correct += (pred == batch_y).float().sum().item()
        epoch_loss += loss.item() * batch_x.size(0)
        n_samples += batch_y.size(0)

    avg_loss = epoch_loss / n_samples
    avg_acc = epoch_correct / n_samples
    loss_history.append(avg_loss)
    acc_history.append(avg_acc)

    if (epoch + 1) % 200 == 0:
        print(f"epoch={epoch+1:5d}  loss={avg_loss:.4f}  acc={avg_acc:.4f}")

print("训练结束。")

## 5. 可视化

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(acc_history, color="orange")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training Accuracy")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()